<a href="https://colab.research.google.com/github/mhmdasln13-create/MLP-Enhancer/blob/main/mlp_enhancer_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Causal MLP ile konuşma iyileştirme (NumPy)
Sıra: **1) veri indir → 2) Ayarlar → 3) STFT → 4) Veri kaynakları → 5) Metrikler → 6) MLP eğitimi**. Hücreleri yukarıdan aşağı sırayla çalıştırın (Çalışma zamanı → Tümünü çalıştır).
Oturum sıfırlanırsa (klasörler silinir) 1. hücreden başlayın.

In [ ]:
# 1) VERİ İNDİR (oturum her sıfırlandığında yeniden çalıştırın)
!wget -q https://www.openslr.org/resources/12/dev-clean.tar.gz
!tar -xzf dev-clean.tar.gz
!wget -q -O esc50.zip https://github.com/karolpiczak/ESC-50/archive/master.zip
!unzip -q esc50.zip
!pip -q install soundfile pystoi

import glob, os, shutil, numpy as np, soundfile as sf
os.makedirs("/content/dns_clean", exist_ok=True)
os.makedirs("/content/dns_noise", exist_ok=True)
rng = np.random.default_rng(0)

flacs = sorted(glob.glob("/content/LibriSpeech/dev-clean/**/*.flac", recursive=True))
for p in rng.choice(flacs, 300, replace=False):
    x, fs = sf.read(str(p))
    sf.write("/content/dns_clean/" + os.path.basename(p).replace(".flac", ".wav"), x, fs)

wavs = sorted(glob.glob("/content/ESC-50-master/audio/*.wav"))
for p in rng.choice(wavs, 400, replace=False):
    shutil.copy(str(p), "/content/dns_noise/")

print(len(os.listdir("/content/dns_clean")), "temiz,", len(os.listdir("/content/dns_noise")), "gürültü")

300 temiz, 400 gürültü


In [ ]:
import os, re, glob, time
import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly
import matplotlib.pyplot as plt

CFG = dict(
    DATA_MODE = "mix",                      # "dns_pairs" | "mix" | "synthetic"
    DNS_ROOT  = "/content/drive/MyDrive/ndns",   # dns_pairs: training_set/ ve validation_set/ içerir
    CLEAN_DIR = "/content/dns_clean",        # mix modu
    NOISE_DIR = "/content/dns_noise",        # mix modu
    MAX_FILES = 500,                         # RAM'e yüklenecek en fazla dosya (her biri ~2 MB / 30 sn)
    SNR_RANGE = (-5.0, 15.0),                # eğitim SNR aralığı (dB)

    FS=16000, N_FFT=512, HOP=128,            # 512/16k = 32 ms algoritmik gecikme (hedef <= 40 ms)
    H1=128, H2=128,                          # gizli nöron sayıları
    BETA1=0.75, BETA2=0.875, ALPHA=0.5,      # sızıntı (1-1/4, 1-1/8, 1/2: donanımda shift ile yapılabilir)
    TH=1.0, SG_K=4.0,                        # eşik, surrogate eğimi
    SEG_SEC=1.0, BATCH=16,
    ITERS=int(os.environ.get("ITERS", 6000)),
    LR=2e-3, CLIP=5.0,
    RATE_REG=1e-2,                           # ateşleme oranı cezası (güç vekili); 0 yaparsanız kapanır
    SEED=0,
)
FS, N_FFT, HOP = CFG["FS"], CFG["N_FFT"], CFG["HOP"]
NF = N_FFT // 2 + 1
H1, H2 = CFG["H1"], CFG["H2"]
BETA1, BETA2, ALPHA, TH, SG_K = CFG["BETA1"], CFG["BETA2"], CFG["ALPHA"], CFG["TH"], CFG["SG_K"]
rng = np.random.default_rng(CFG["SEED"])
print("Algoritmik gecikme: %.1f ms, frame hızı: %.1f frame/s" % (1000 * N_FFT / FS, FS / HOP))

Algoritmik gecikme: 32.0 ms, frame hızı: 125.0 frame/s


In [ ]:
WIN = np.sqrt(np.hanning(N_FFT + 1)[:-1]).astype(np.float32)   # sqrt-Hann, hop=N/4 → /2 ile mükemmel yeniden yapılandırma

def stft(x):
    """Causal: frame t yalnızca (t+1)*HOP örneğine kadar olan veriyi kullanır."""
    x = x.astype(np.float32)
    tail = (-len(x)) % HOP + (N_FFT - HOP)                    # sona sıfır: OLA kuyruğunu tamamla
    xp = np.concatenate([np.zeros(N_FFT - HOP, np.float32), x, np.zeros(tail, np.float32)])
    T = (len(xp) - N_FFT) // HOP + 1
    idx = np.arange(N_FFT)[None, :] + HOP * np.arange(T)[:, None]
    return np.fft.rfft(xp[idx] * WIN, axis=1).astype(np.complex64)

def istft(X, length):
    fr = np.fft.irfft(X, n=N_FFT, axis=1).astype(np.float32) * WIN
    T = fr.shape[0]
    out = np.zeros(HOP * (T - 1) + N_FFT, np.float32)
    for t in range(T):
        out[t * HOP:t * HOP + N_FFT] += fr[t]
    return out[N_FFT - HOP:N_FFT - HOP + length] / 2.0

_x = rng.standard_normal(5000).astype(np.float32) * 0.1
assert np.abs(istft(stft(_x), len(_x)) - _x).max() < 1e-4, "STFT yeniden yapılandırma hatası"
print("STFT/ISTFT yeniden yapılandırma OK")

def feats(Ym):
    """Sabit (veriden bağımsız) log-sıkıştırma: donanımda LUT ile yapılabilir."""
    return np.log1p(20.0 * Ym).astype(np.float32)

STFT/ISTFT yeniden yapılandırma OK


In [ ]:
def load_wav(path):
    fs, x = wavfile.read(path)
    if x.dtype == np.int16:   x = x / 32768.0
    elif x.dtype == np.int32: x = x / 2147483648.0
    else:                     x = x.astype(np.float32)
    if x.ndim > 1: x = x.mean(1)
    if fs != FS:   x = resample_poly(x, FS, fs)
    return x.astype(np.float32)

def crop(x, L, rng):
    if len(x) < L:
        return np.pad(x, (0, L - len(x)))
    s = rng.integers(0, len(x) - L + 1)
    return x[s:s + L]

def mix_at_snr(s, n, snr_db):
    ps, pn = np.mean(s ** 2) + 1e-10, np.mean(n ** 2) + 1e-10
    n = n * np.sqrt(ps / (pn * 10 ** (snr_db / 10)))
    y = s + n
    g = 0.95 / (np.abs(y).max() + 1e-8)                       # kırpılmayı önle
    return (y * g).astype(np.float32), (s * g).astype(np.float32)

class PairSource:
    """DNS'in hazır noisy/clean çiftleri (noisyspeech_synthesizer çıktısı)."""
    def __init__(self, root, max_files):
        noisy = sorted(glob.glob(os.path.join(root, "noisy", "*.wav")))
        clean = sorted(glob.glob(os.path.join(root, "clean", "*.wav")))
        assert noisy and clean, f"{root}/noisy ve /clean içinde wav bulunamadı"
        fid = lambda p: (re.search(r"fileid_(\d+)", os.path.basename(p)) or [None, None])[1]
        cmap = {fid(p): p for p in clean if fid(p) is not None}
        pairs = [(n, cmap[fid(n)]) for n in noisy if fid(n) in cmap]
        if not pairs:                                          # fileid yoksa sıralı eşle
            pairs = list(zip(noisy, clean))
        pairs = pairs[:max_files]
        print(f"{root}: {len(pairs)} çift yükleniyor...")
        self.data = []
        for n, c in pairs:
            xn, xc = load_wav(n), load_wav(c)
            m = min(len(xn), len(xc))
            self.data.append((xn[:m], xc[:m]))
    def sample(self, B, L, rng):
        out_n, out_c = [], []
        for _ in range(B):
            xn, xc = self.data[rng.integers(len(self.data))]
            if len(xn) >= L:
                s = rng.integers(0, len(xn) - L + 1); out_n.append(xn[s:s + L]); out_c.append(xc[s:s + L])
            else:
                out_n.append(np.pad(xn, (0, L - len(xn)))); out_c.append(np.pad(xc, (0, L - len(xc))))
        return np.stack(out_n), np.stack(out_c)

class MixSource:
    """Temiz konuşma + gürültü listesini eğitim sırasında rastgele SNR ile karıştırır."""
    def __init__(self, clean_list, noise_list, snr_range):
        self.clean, self.noise, self.snr = clean_list, noise_list, snr_range
    def sample(self, B, L, rng):
        yn, yc = [], []
        for _ in range(B):
            s = crop(self.clean[rng.integers(len(self.clean))], L, rng)
            n = crop(self.noise[rng.integers(len(self.noise))], L, rng)
            a, b = mix_at_snr(s, n, rng.uniform(*self.snr))
            yn.append(a); yc.append(b)
        return np.stack(yn), np.stack(yc)

def load_dir(d, max_files):
    files = sorted(glob.glob(os.path.join(d, "**", "*.wav"), recursive=True))
    np.random.default_rng(0).shuffle(files); files = files[:max_files]
    assert files, f"{d} içinde wav yok"
    return [load_wav(f) for f in files]

# --- sentetik yedek veri ---
def synth_speech(L, rng):
    t = (np.arange(L) / FS).astype(np.float32)
    f0 = rng.uniform(90, 220) * (1 + 0.15 * np.sin(2 * np.pi * rng.uniform(0.3, 1.2) * t + rng.uniform(0, 6.28)))
    f0 = f0.astype(np.float32)
    phase = 2 * np.pi * np.cumsum(f0) / FS
    K = int(FS / 2 * 0.9 // f0.max())
    k = np.arange(1, K + 1, dtype=np.float32)[:, None]
    fk = k * f0[None, :]
    env = np.zeros_like(fk)
    for fc, bw in [(rng.uniform(300, 800), 120), (rng.uniform(900, 2200), 200), (rng.uniform(2300, 3300), 250)]:
        fcc = fc * (1 + 0.15 * np.sin(2 * np.pi * rng.uniform(1, 3) * t + rng.uniform(0, 6.28)))[None, :]
        env += np.exp(-0.5 * ((fk - fcc) / bw) ** 2)
    sig = (env / np.sqrt(k) * np.sin(k * phase[None, :])).sum(0)
    gate = 1 / (1 + np.exp(-8 * (np.sin(2 * np.pi * rng.uniform(2.5, 4.5) * t + rng.uniform(0, 6.28)) - 0.1)))
    sig = sig * gate
    return (sig / (np.sqrt(np.mean(sig ** 2)) + 1e-8) * 0.1).astype(np.float32)

def synth_noise(L, rng):
    w = rng.standard_normal(L).astype(np.float32)
    Wf = np.fft.rfft(w)
    f = np.fft.rfftfreq(L, 1 / FS); f[0] = f[1]
    n = np.fft.irfft(Wf / f ** (rng.uniform(0, 2) / 2), n=L).astype(np.float32)
    t = np.arange(L) / FS
    if rng.random() < 0.5:
        n += rng.uniform(0.3, 1.0) * n.std() * np.sin(2 * np.pi * rng.uniform(100, 1500) * t)
    if rng.random() < 0.5:                                      # kesikli patlamalar
        g = (np.sin(2 * np.pi * rng.uniform(1, 6) * t) > 0.6).astype(np.float32)
        n += g * rng.standard_normal(L).astype(np.float32) * n.std() * 2
    return (n / (n.std() + 1e-8) * 0.1).astype(np.float32)

def make_source(split, rng):
    mode = CFG["DATA_MODE"]
    if mode == "dns_pairs":
        sub = "training_set" if split == "train" else "validation_set"
        return PairSource(os.path.join(CFG["DNS_ROOT"], sub), CFG["MAX_FILES"])
    if mode == "mix":
        cl, no = load_dir(CFG["CLEAN_DIR"], CFG["MAX_FILES"]), load_dir(CFG["NOISE_DIR"], CFG["MAX_FILES"])
        ncl, nno = max(1, int(0.9 * len(cl))), max(1, int(0.9 * len(no)))
        cl = cl[:ncl] if split == "train" else cl[ncl:] or cl[-1:]
        no = no[:nno] if split == "train" else no[nno:] or no[-1:]
        return MixSource(cl, no, CFG["SNR_RANGE"])
    # synthetic
    print(f"Sentetik havuz üretiliyor ({split})...")
    nc, nn = (60, 30) if split == "train" else (12, 8)
    L = 3 * FS
    return MixSource([synth_speech(L, rng) for _ in range(nc)], [synth_noise(L, rng) for _ in range(nn)], CFG["SNR_RANGE"])

t0 = time.time()
src_train = make_source("train", rng)
src_val = make_source("val", rng)
SEG = int(CFG["SEG_SEC"] * FS)
val_noisy, val_clean = src_val.sample(24, 3 * FS, np.random.default_rng(123))
print("veri hazır (%.1f s) | val: %s" % (time.time() - t0, val_noisy.shape))

veri hazır (17.5 s) | val: (24, 48000)


In [ ]:
def si_sdr(est, ref, eps=1e-8):
    ref = ref - ref.mean(); est = est - est.mean()
    t = (est @ ref) / (ref @ ref + eps) * ref
    e = est - t
    return 10 * np.log10((t @ t + eps) / (e @ e + eps))

try:
    from pystoi import stoi as _stoi        # Colab'da: !pip -q install pystoi
except Exception:
    _stoi = None

def enhance_batch(P, noisy):
    f, _, _, Y = prep_batch(noisy, noisy)
    M, cache = forward(P, f)
    return np.stack([istft(M[i] * Y[i], noisy.shape[1]) for i in range(len(noisy))]), cache

def evaluate(P, noisy, clean, with_stoi=False):
    est, cache = enhance_batch(P, noisy)
    sn = np.array([si_sdr(noisy[i], clean[i]) for i in range(len(noisy))])
    se = np.array([si_sdr(est[i], clean[i]) for i in range(len(noisy))])
    r = dict(sisdr_noisy=sn.mean(), sisdr_enh=se.mean(), sisdri=(se - sn).mean(),
             rate1=cache["S1"].mean(), rate2=cache["S2"].mean())
    if with_stoi and _stoi is not None:
        r["stoi_noisy"] = np.mean([_stoi(clean[i], noisy[i], FS) for i in range(len(noisy))])
        r["stoi_enh"] = np.mean([_stoi(clean[i], est[i], FS) for i in range(len(noisy))])
    return r

In [ ]:
def _sample(self, B, L, rng):
    yn, yc = [], []
    for _ in range(B):
        for _try in range(30):                      # sessiz konuşma parçalarını ele
            s = crop(self.clean[rng.integers(len(self.clean))], L, rng)
            if np.sqrt(np.mean(s ** 2)) > 1e-3: break
        for _try in range(30):                      # sessiz / çok seyrek gürültü parçalarını ele
            n = crop(self.noise[rng.integers(len(self.noise))], L, rng)
            if np.sqrt(np.mean(n ** 2)) > 1e-3 and (np.abs(n) > 1e-4).mean() > 0.5: break
        a, b = mix_at_snr(s, n, rng.uniform(*self.snr))
        yn.append(a); yc.append(b)
    return np.stack(yn), np.stack(yc)

MixSource.sample = _sample
val_noisy, val_clean = src_val.sample(24, 3 * FS, np.random.default_rng(123))
print("val gürültülü SI-SDR ortalaması:", np.mean([si_sdr(val_noisy[i], val_clean[i]) for i in range(24)]))

val gürültülü SI-SDR ortalaması: 5.2585564


In [ ]:
# ===== MLP BASELINE (saf NumPy, causal) =====
# Yapıştırma yeri: "Metrikler ve değerlendirme" hücresinden SONRA yeni bir hücre.
# Gereken önceki hücreler: Ayarlar, STFT, Veri kaynakları, Metrikler (SNN eğitim hücrelerini çalıştırmanıza gerek yok).
# Giriş: şimdiki + önceki K-1 frame'in log-magnitude'u (gelecek frame yok → ek gecikme yok, 32 ms aynı).

MLP = dict(K=6, H=256, ITERS=6000, LR=1e-3, BATCH=16, B3_INIT=2.0)

def mlp_ctx(f, K):
    """f: (B,T,NF) -> (B*T, K*NF): her satır [f_t, f_{t-1}, ..., f_{t-K+1}] (başta sıfır dolgu)."""
    B, T, D = f.shape
    pad = np.concatenate([np.zeros((B, K - 1, D), np.float32), f], axis=1)
    return np.concatenate([pad[:, K - 1 - k:K - 1 - k + T] for k in range(K)], axis=2).reshape(B * T, K * D)

def mlp_prep(noisy, clean):
    Y = np.stack([stft(x) for x in noisy]); S = np.stack([stft(x) for x in clean])
    Ym, Sm = np.abs(Y), np.abs(S)
    irm = np.clip(Sm / (Ym + 1e-8), 0, 1).astype(np.float32)
    w = np.minimum(Ym / (Ym.mean() + 1e-8), 10.0)            # enerji ağırlığı (SI-SDR ile uyumlu)
    w = (w / w.mean()).astype(np.float32)
    return feats(Ym), irm, w, Y

def mlp_init(rng, D, H, NFo):
    he = lambda i, o: (rng.standard_normal((i, o)) * np.sqrt(2.0 / i)).astype(np.float32)
    return dict(W1=he(D, H), b1=np.zeros(H, np.float32), W2=he(H, H), b2=np.zeros(H, np.float32),
                W3=(rng.standard_normal((H, NFo)) * 0.05).astype(np.float32), b3=np.full(NFo, MLP["B3_INIT"], np.float32))

def mlp_forward(Pm, X):
    Z1 = X @ Pm["W1"] + Pm["b1"]; A1 = np.maximum(Z1, 0)
    Z2 = A1 @ Pm["W2"] + Pm["b2"]; A2 = np.maximum(Z2, 0)
    M = 1.0 / (1.0 + np.exp(-(A2 @ Pm["W3"] + Pm["b3"])))
    return M.astype(np.float32), (X, Z1, A1, Z2, A2)

def mlp_backward(Pm, c, M, irm, w):
    X, Z1, A1, Z2, A2 = c
    err = M - irm
    loss = float(np.mean(w * err ** 2))
    dZ3 = (2 * w * err / M.size) * M * (1 - M)
    G = {"W3": A2.T @ dZ3, "b3": dZ3.sum(0)}
    dZ2 = (dZ3 @ Pm["W3"].T) * (Z2 > 0)
    G["W2"] = A1.T @ dZ2; G["b2"] = dZ2.sum(0)
    dZ1 = (dZ2 @ Pm["W2"].T) * (Z1 > 0)
    G["W1"] = X.T @ dZ1; G["b1"] = dZ1.sum(0)
    return loss, G

class AdamM:
    def __init__(self, Pm, lr): self.lr, self.t = lr, 0; self.m = {k: np.zeros_like(v) for k, v in Pm.items()}; self.v = {k: np.zeros_like(v) for k, v in Pm.items()}
    def step(self, Pm, G):
        self.t += 1
        for k in Pm:
            self.m[k] = 0.9 * self.m[k] + 0.1 * G[k]; self.v[k] = 0.999 * self.v[k] + 0.001 * G[k] ** 2
            Pm[k] -= self.lr * (self.m[k] / (1 - 0.9 ** self.t)) / (np.sqrt(self.v[k] / (1 - 0.999 ** self.t)) + 1e-8)

def mlp_enhance(Pm, noisy):
    f, _, _, Y = mlp_prep(noisy, noisy)
    M, _ = mlp_forward(Pm, mlp_ctx(f, MLP["K"]))
    M = M.reshape(f.shape)
    return np.stack([istft(M[i] * Y[i], noisy.shape[1]) for i in range(len(noisy))]), M

def mlp_eval(Pm, noisy, clean):
    est, M = mlp_enhance(Pm, noisy)
    sn = np.array([si_sdr(noisy[i], clean[i]) for i in range(len(noisy))])
    se = np.array([si_sdr(est[i], clean[i]) for i in range(len(noisy))])
    return sn.mean(), se.mean(), (se - sn).mean()

Pm = mlp_init(np.random.default_rng(CFG["SEED"]), MLP["K"] * NF, MLP["H"], NF)
optm = AdamM(Pm, MLP["LR"]); mrng = np.random.default_rng(CFG["SEED"] + 1)
mh = []; t0 = time.time()
best = (-1e9, None)
print("MLP parametre sayısı:", sum(v.size for v in Pm.values()))
for it in range(1, MLP["ITERS"] + 1):
    optm.lr = MLP["LR"] * min(1.0, it / 200) * (0.05 + 0.95 * 0.5 * (1 + np.cos(np.pi * it / MLP["ITERS"])))
    noisy, clean = src_train.sample(MLP["BATCH"], SEG, mrng)
    f, irm, w, _ = mlp_prep(noisy, clean)
    X = mlp_ctx(f, MLP["K"])
    M, c = mlp_forward(Pm, X)
    loss, G = mlp_backward(Pm, c, M, irm.reshape(-1, NF), w.reshape(-1, NF))
    optm.step(Pm, G); mh.append(loss)
    if it % 100 == 0 or it == 1:
        print(f"it {it:5d} | loss {np.mean(mh[-100:]):.4f} | {time.time()-t0:.0f}s")
    if it % 500 == 0 or it == MLP["ITERS"]:
        a, b, d = mlp_eval(Pm, val_noisy, val_clean)
        print(f"   >> val SI-SDR: noisy {a:.2f} → enh {b:.2f} dB (iyileşme {d:+.2f} dB)")
        if d > best[0]: best = (d, {k: v.copy() for k, v in Pm.items()})

Pm = best[1]; print("en iyi val iyileşmesi: %+.2f dB (bu ağırlıklar kullanılıyor)" % best[0])
# --- girişe bağlı maske korelasyonu (≈0 ise model sabit bir maske öğrenmiştir) ---
f, irm, w, Y = mlp_prep(val_noisy, val_clean)
Mv = mlp_forward(Pm, mlp_ctx(f, MLP["K"]))[0].reshape(f.shape)
dm, di = Mv - Mv.mean((0, 1)), irm - irm.mean((0, 1))
print("girişe bağlı maske korelasyonu:", round(float((dm * di).sum() / np.sqrt((dm ** 2).sum() * (di ** 2).sum())), 3))

# --- INT8 PTQ (ağırlıklar) ---
def q8(W):
    s = float(np.abs(W).max()) / 127.0 + 1e-12
    return np.clip(np.round(W / s), -127, 127).astype(np.int8), s
Pq = {k: (q8(v)[0].astype(np.float32) * q8(v)[1] if k.startswith("W") else v.copy()) for k, v in Pm.items()}
print("FP32 :", [round(float(x), 3) for x in mlp_eval(Pm, val_noisy, val_clean)])
print("INT8 :", [round(float(x), 3) for x in mlp_eval(Pq, val_noisy, val_clean)])
print("Ağırlık belleği: %.0f KB → %.0f KB" % (sum(v.nbytes for k, v in Pm.items() if k[0] == "W") / 1024,
                                                sum(v.size for k, v in Pm.items() if k[0] == "W") / 1024))
np.savez("mlp_enhancer_fp32.npz", **Pm)


MLP parametre sayısı: 526849
it     1 | loss 0.2592 | 0s
it   100 | loss 0.2079 | 24s
it   200 | loss 0.1642 | 42s
it   300 | loss 0.1550 | 61s
it   400 | loss 0.1519 | 81s
it   500 | loss 0.1511 | 98s
   >> val SI-SDR: noisy 5.26 → enh 7.39 dB (iyileşme +2.13 dB)
it   600 | loss 0.1499 | 117s
it   700 | loss 0.1425 | 137s
it   800 | loss 0.1448 | 154s
it   900 | loss 0.1390 | 173s
it  1000 | loss 0.1380 | 192s
   >> val SI-SDR: noisy 5.26 → enh 7.55 dB (iyileşme +2.29 dB)
it  1100 | loss 0.1376 | 211s
it  1200 | loss 0.1337 | 231s
it  1300 | loss 0.1350 | 250s
it  1400 | loss 0.1324 | 268s
it  1500 | loss 0.1307 | 288s
   >> val SI-SDR: noisy 5.26 → enh 8.20 dB (iyileşme +2.94 dB)
it  1600 | loss 0.1272 | 307s
it  1700 | loss 0.1271 | 327s
it  1800 | loss 0.1248 | 346s
it  1900 | loss 0.1237 | 365s
it  2000 | loss 0.1248 | 385s
   >> val SI-SDR: noisy 5.26 → enh 7.84 dB (iyileşme +2.59 dB)
it  2100 | loss 0.1261 | 406s
it  2200 | loss 0.1233 | 426s
it  2300 | loss 0.1223 | 444s
it  24

In [ ]:
# ===== TEŞHİS HÜCRESİ (MLP eğitiminden sonra çalıştırın) =====
def snr_report(name, noisy, clean):
    est, _ = mlp_enhance(Pm, noisy)
    sn = np.array([si_sdr(noisy[i], clean[i]) for i in range(len(noisy))])
    se = np.array([si_sdr(est[i], clean[i]) for i in range(len(noisy))])
    print(name, "| ort. iyileşme: %+.2f dB" % (se - sn).mean())
    for lo, hi in [(-99, 0), (0, 5), (5, 10), (10, 15), (15, 99)]:
        m = (sn >= lo) & (sn < hi)
        if m.any():
            print(f"   gürültülü SI-SDR [{lo:>3},{hi:>3}) n={m.sum():2d}: iyileşme {np.mean(se[m] - sn[m]):+.2f} dB")

tr_n, tr_c = src_train.sample(48, 3 * FS, np.random.default_rng(7))
snr_report("EĞİTİM verisi (48 klip)", tr_n, tr_c)
snr_report("VAL verisi (24 klip)", val_noisy, val_clean)
print("gürültü dosyası sayısı -> train:", len(src_train.noise), "| val:", len(src_val.noise))

f, irm, w, Y = mlp_prep(val_noisy, val_clean)
M_ = mlp_forward(Pm, mlp_ctx(f, MLP["K"]))[0].reshape(f.shape)
L = lambda m: float(np.mean(w * (m - irm) ** 2))
print("val kaybı | maske=1: %.4f | bin-ortalamalı sabit maske: %.4f | model: %.4f"
      % (L(np.ones_like(irm)), L(np.broadcast_to(irm.mean((0, 1)), irm.shape)), L(M_)))

EĞİTİM verisi (48 klip) | ort. iyileşme: +2.76 dB
   gürültülü SI-SDR [-99,  0) n=12: iyileşme +3.86 dB
   gürültülü SI-SDR [  0,  5) n=10: iyileşme +4.50 dB
   gürültülü SI-SDR [  5, 10) n=15: iyileşme +2.38 dB
   gürültülü SI-SDR [ 10, 15) n=11: iyileşme +0.48 dB
VAL verisi (24 klip) | ort. iyileşme: +3.87 dB
   gürültülü SI-SDR [-99,  0) n= 5: iyileşme +9.60 dB
   gürültülü SI-SDR [  0,  5) n= 8: iyileşme +3.00 dB
   gürültülü SI-SDR [  5, 10) n= 4: iyileşme +2.56 dB
   gürültülü SI-SDR [ 10, 15) n= 7: iyileşme +1.54 dB
gürültü dosyası sayısı -> train: 360 | val: 40
val kaybı | maske=1: 0.3664 | bin-ortalamalı sabit maske: 0.1780 | model: 0.1030


In [ ]:
# ===== SES ÇIKTISI: dinle + wav olarak kaydet (MLP eğitiminden sonra) =====
from IPython.display import Audio, display

def to_wav(name, x):
    wavfile.write(name, FS, (np.clip(x, -1, 1) * 32767).astype(np.int16))

idx = [0, 1, 2, 3]                                   # dinlemek istediğiniz val klipleri
est, _ = mlp_enhance(Pm, val_noisy[idx])
for k, i in enumerate(idx):
    print(f"=== klip {i} | SI-SDR: gürültülü {si_sdr(val_noisy[i], val_clean[i]):.1f} dB → iyileştirilmiş {si_sdr(est[k], val_clean[i]):.1f} dB")
    for etiket, dosya, x in [("Gürültülü", "noisy", val_noisy[i]), ("İyileştirilmiş", "enhanced", est[k]), ("Temiz (referans)", "clean", val_clean[i])]:
        print(etiket); display(Audio(np.clip(x, -1, 1), rate=FS, normalize=False))
        to_wav(f"klip{i}_{dosya}.wav", x)
print("wav dosyaları Colab'ın sol 'Dosyalar' panelinde (klipN_noisy/enhanced/clean.wav)")

# --- Kendi kaydınız: dosya yükleyip iyileştirin (isteğe bağlı) ---
# from google.colab import files
# up = files.upload(); ad = list(up)[0]
# x = load_wav(ad)                                   # mono ve 16 kHz'e çevirir
# y = mlp_enhance(Pm, x[None])[0][0]
# print("Gürültülü");       display(Audio(np.clip(x, -1, 1), rate=FS, normalize=False))
# print("İyileştirilmiş");  display(Audio(np.clip(y, -1, 1), rate=FS, normalize=False))
# to_wav("benim_enhanced.wav", y); files.download("benim_enhanced.wav")

=== klip 0 | SI-SDR: gürültülü -1.3 dB → iyileştirilmiş 8.1 dB
Gürültülü


İyileştirilmiş


Temiz (referans)


=== klip 1 | SI-SDR: gürültülü 13.5 dB → iyileştirilmiş 14.3 dB
Gürültülü


İyileştirilmiş


Temiz (referans)


... (3 outputs left out)

=== klip 3 | SI-SDR: gürültülü 11.5 dB → iyileştirilmiş 12.2 dB
Gürültülü


İyileştirilmiş


Temiz (referans)


wav dosyaları Colab'ın sol 'Dosyalar' panelinde (klipN_noisy/enhanced/clean.wav)
